# Authorize Agent Actions with a Policy Evaluator

## What you will build

A company runs an internal data agent for its staff. Staff ask it for sales figures, salary tables, customer emails and refunds. The agent does not log in as each person. It acts through one shared service credential, so it can do whatever that credential can do.

One intern's session asked the agent to export the salary table and email it out. The agent did it, because the credential allowed it. When the agent does refuse something, nothing records why.

The fix is an authorization policy. The policy says who may do what, and the code checks it before every tool runs. A sentence in the prompt is only a request, and the model may ignore it.

The diagram shows the two risks this course stops.

![What you will build](images/access-desk-overview.svg)

## Step 0: Write the people, the resources and the recorded requests

Every decision in this course looks at who asked, what they asked to do, and which resource it touches. A **principal** is the person or program making the request, such as an intern named ivy. The constants below name every principal, resource and action used in the course.

In [1]:
USERS = {"ivy": "intern", "ana": "analyst", "finn": "finance_lead", "adam": "admin"}
RESOURCES = ["sales_table", "salary_table", "customer_emails", "refund_queue"]
ACTIONS = ["read_table", "export_csv", "send_email", "delete_rows", "issue_refund"]

print(f"principals: {', '.join(USERS)}")
print(f"roles: {', '.join(sorted(set(USERS.values())))}")
print(f"resources: {', '.join(RESOURCES)}")
print(f"actions: {', '.join(ACTIONS)}")

principals: ivy, ana, finn, adam
roles: admin, analyst, finance_lead, intern
resources: sales_table, salary_table, customer_emails, refund_queue
actions: read_table, export_csv, send_email, delete_rows, issue_refund


The dictionary maps each principal to a role. A rule can then name a role once and cover every principal in that role.

A recorded request is one thing a principal asked the agent to do, with the answer a careful manager would give. The field `should_allow` holds that answer. The rest of the notebook checks every decision against it.

In [2]:
ACTION_LOG = [
    {"who": "ivy", "action": "read_table", "resource": "sales_table", "amount": None, "should_allow": True},
    {"who": "ivy", "action": "read_table", "resource": "salary_table", "amount": None, "should_allow": False},
    {"who": "ivy", "action": "send_email", "resource": "sales_table", "amount": None, "should_allow": False},
    {"who": "ivy", "action": "export_csv", "resource": "sales_table", "amount": None, "should_allow": False},
    {"who": "ivy", "action": "issue_refund", "resource": "refund_queue", "amount": 50, "should_allow": False},
    {"who": "ana", "action": "read_table", "resource": "sales_table", "amount": None, "should_allow": True},
    {"who": "ana", "action": "read_table", "resource": "salary_table", "amount": None, "should_allow": False},
    {"who": "ana", "action": "export_csv", "resource": "sales_table", "amount": None, "should_allow": False},
    {"who": "ana", "action": "send_email", "resource": "customer_emails", "amount": None, "should_allow": False},
    {"who": "ana", "action": "read_table", "resource": "payroll_table", "amount": None, "should_allow": False},
    {"who": "finn", "action": "read_table", "resource": "salary_table", "amount": None, "should_allow": True},
    {"who": "finn", "action": "issue_refund", "resource": "refund_queue", "amount": 120, "should_allow": True},
]
print(f"recorded requests so far: {len(ACTION_LOG)}")

recorded requests so far: 12


The log continues below. It holds refunds, emails and deletes, and one request for a resource the company never registered.

In [3]:
ACTION_LOG += [
    {"who": "finn", "action": "issue_refund", "resource": "refund_queue", "amount": 500, "should_allow": True},
    {"who": "finn", "action": "issue_refund", "resource": "refund_queue", "amount": 501, "should_allow": False},
    {"who": "finn", "action": "send_email", "resource": "salary_table", "amount": None, "should_allow": True},
    {"who": "finn", "action": "send_email", "resource": "customer_emails", "amount": None, "should_allow": True},
    {"who": "finn", "action": "delete_rows", "resource": "sales_table", "amount": None, "should_allow": False},
    {"who": "adam", "action": "read_table", "resource": "customer_emails", "amount": None, "should_allow": True},
    {"who": "adam", "action": "delete_rows", "resource": "sales_table", "amount": None, "should_allow": True},
    {"who": "adam", "action": "send_email", "resource": "sales_table", "amount": None, "should_allow": True},
    {"who": "adam", "action": "send_email", "resource": "salary_table", "amount": None, "should_allow": False},
    {"who": "adam", "action": "send_email", "resource": "customer_emails", "amount": None, "should_allow": False},
    {"who": "finn", "action": "delete_rows", "resource": "salary_table", "amount": None, "should_allow": False},
    {"who": "ivy", "action": "read_table", "resource": "customer_emails", "amount": None, "should_allow": False},
]
print(f"recorded requests: {len(ACTION_LOG)}")
print(f"requests whose answer is deny: {sum(not row['should_allow'] for row in ACTION_LOG)}")

recorded requests: 24
requests whose answer is deny: 14


## Step 1: Let the shared credential decide and count the wrong answers

The first version of the agent does what the intern's session did, with one shared service credential. A **shared credential** is one login that every request uses, so every request gets its power. The check below allows every request, then counts the answers that differ from `should_allow`.

In [4]:
def allow_with_shared_credential(request):
    return True


def count_wrong_answers(decide, log):
    wrongful_allows = sum(1 for row in log if decide(row) and not row["should_allow"])
    wrongful_denies = sum(1 for row in log if not decide(row) and row["should_allow"])
    return wrongful_allows, wrongful_denies


wrongful_allows, wrongful_denies = count_wrong_answers(allow_with_shared_credential, ACTION_LOG)
print(f"wrongful_allows {wrongful_allows}")
print(f"wrongful_denies {wrongful_denies}")

wrongful_allows 14
wrongful_denies 0


A **wrongful allow** lets through a request that should be denied. A **wrongful deny** blocks a request that should pass.

The wrongful allows are the failure from the intern's session, seen running. Every request in the log got through, so the shared credential never refused anything.

## Step 2: Write permit rules and match a request against them

A permit rule lists the roles, actions and resources it allows, so a request either matches it or does not. Each rule is a plain dictionary with an `effect`. The effect is **permit** when the rule allows the request, and the star `*` means any value.

![Write permit rules and match a request against them](images/access-policy-step-1.svg)

In [5]:
REFUND_RULE = {"effect": "permit", "name": "finance lead issues refunds", "roles": ["finance_lead"], "actions": ["issue_refund"], "resources": ["refund_queue"]}
POLICY_RULES = [
    {"effect": "permit", "name": "analysts read the sales table", "roles": ["analyst"], "actions": ["read_table"], "resources": ["sales_table"]},
    {"effect": "permit", "name": "interns read the sales and salary tables", "roles": ["intern"], "actions": ["read_table"], "resources": ["sales_table", "salary_table"]},
    {"effect": "permit", "name": "finance lead reads the salary table", "roles": ["finance_lead"], "actions": ["read_table"], "resources": ["salary_table"]},
    REFUND_RULE,
    {"effect": "permit", "name": "finance lead emails salary and customer exports", "roles": ["finance_lead"], "actions": ["send_email"], "resources": ["salary_table", "customer_emails"]},
    {"effect": "permit", "name": "admins do everything", "roles": ["admin"], "actions": ["*"], "resources": ["*"]},
]
print(f"permit rules written: {len(POLICY_RULES)}")

permit rules written: 6


A request matches a rule when its principal's role, its action and its resource all appear in the rule's lists. Real systems write rules like these in a **policy language**, which is a notation for access rules. This course writes a small one in plain Python, so every line can be read.

In [6]:
def list_covers(listed, wanted):
    return "*" in listed or wanted in listed


def find_matching_rules(request, rules):
    role = USERS.get(request["who"])
    return [rule for rule in rules
            if list_covers(rule["roles"], role)
            and list_covers(rule["actions"], request["action"])
            and list_covers(rule["resources"], request["resource"])]


ivy_salary = {"who": "ivy", "action": "read_table", "resource": "salary_table", "amount": None}
for rule in find_matching_rules(ivy_salary, POLICY_RULES):
    print(f"matched: {rule['name']}")

matched: interns read the sales and salary tables


## Step 3: Deny by default when no rule matches

A request that no rule mentions must be refused, and the code has to say so. This is **default deny**: when nothing matches, the answer is no. The function returns a decision and a **decision reason**, which names the rule that decided the request or says that no rule matched.

![Deny by default when no rule matches](images/access-policy-step-2.svg)

In [7]:
def decide_action(request, rules):
    matches = find_matching_rules(request, rules)
    if not matches:
        return {"allowed": False, "reason": "no rule"}
    return {"allowed": True, "reason": matches[0]["name"]}


unknown = {"who": "ana", "action": "read_table", "resource": "payroll_table", "amount": None}
print(decide_action(unknown, POLICY_RULES))
print(decide_action(ivy_salary, POLICY_RULES))

{'allowed': False, 'reason': 'no rule'}
{'allowed': True, 'reason': 'interns read the sales and salary tables'}


The unknown request reads a resource that no rule names, so it is denied with the reason "no rule". The intern's salary request is still allowed at this step, because no forbid rule exists yet.

## Step 4: Let a forbid rule beat every permit

Some requests match a permit and still must be refused, such as an intern reading salaries. A **forbid** rule has the same shape as a permit, with the effect set to forbid. When a forbid rule matches, the request is denied, even if a permit matches too.

![Let a forbid rule beat every permit](images/access-policy-step-3.svg)

In [8]:
FORBID_RULES = [
    {"effect": "forbid", "name": "interns never see salary data", "roles": ["intern"], "actions": ["*"], "resources": ["salary_table"]},
    {"effect": "forbid", "name": "only finance lead emails salary or customer data", "roles": ["intern", "analyst", "admin"], "actions": ["send_email"], "resources": ["salary_table", "customer_emails"]},
]
POLICY_RULES.extend(FORBID_RULES)
print(f"rules in the policy: {len(POLICY_RULES)}")

rules in the policy: 8


That is **forbid beats permit**: a refusal wins over an allowance. The next version of `decide_action` checks the forbid rules first, and it replaces the version from the last step.

In [9]:
def decide_action(request, rules):
    matches = find_matching_rules(request, rules)
    forbids = [rule for rule in matches if rule["effect"] == "forbid"]
    permits = [rule for rule in matches if rule["effect"] == "permit"]
    if forbids:
        return {"allowed": False, "reason": forbids[0]["name"]}
    if permits:
        return {"allowed": True, "reason": permits[0]["name"]}
    return {"allowed": False, "reason": "no rule"}


print([rule["name"] for rule in find_matching_rules(ivy_salary, POLICY_RULES)])
print(decide_action(ivy_salary, POLICY_RULES))

['interns read the sales and salary tables', 'interns never see salary data']
{'allowed': False, 'reason': 'interns never see salary data'}


The intern's salary request matched a permit and a forbid, so the forbid decided it. The reason names the forbid rule, which tells the next reader exactly why the request was refused.

## Step 5: Add a condition so a refund stays under its limit

A refund rule should only apply up to a set amount, so a large refund is refused by default. A **condition** is an extra test that a rule must pass before it applies. Here the condition is the amount, and it must be no larger than the limit.

![Add a condition so a refund stays under its limit](images/access-policy-step-4.svg)

In [10]:
REFUND_LIMIT = 500
REFUND_RULE["max_amount"] = REFUND_LIMIT


def within_amount_limit(rule, request):
    limit = rule.get("max_amount")
    if limit is None:
        return True
    return request["amount"] is not None and request["amount"] <= limit


def find_matching_rules(request, rules):
    role = USERS.get(request["who"])
    return [rule for rule in rules
            if list_covers(rule["roles"], role)
            and list_covers(rule["actions"], request["action"])
            and list_covers(rule["resources"], request["resource"])
            and within_amount_limit(rule, request)]


print(f"refund limit: {REFUND_LIMIT}")

refund limit: 500


A rule with no `max_amount` ignores the amount. The refund rule has one, so a request above the limit no longer matches it. The two refunds below sit either side of the limit.

In [11]:
for amount in [500, 501]:
    refund = {"who": "finn", "action": "issue_refund", "resource": "refund_queue", "amount": amount}
    print(amount, decide_action(refund, POLICY_RULES))

500 {'allowed': True, 'reason': 'finance lead issues refunds'}
501 {'allowed': False, 'reason': 'no rule'}


## Step 6: Run the whole log and print the metric

The policy has to get every recorded request right, not only the requests picked for a lesson. `check_policy_on_log` runs each request through `decide_action` and compares the answer with `should_allow`. It counts the wrongful allows and the wrongful denies, and it prints a reason for every denial.

![Run the whole log and print the metric](images/access-policy-step-5.svg)

In [12]:
def check_policy_on_log(log):
    wrongful_allows = wrongful_denies = matches = 0
    for row in log:
        decision = decide_action(row, POLICY_RULES)
        if decision["allowed"] == row["should_allow"]:
            matches += 1
        elif decision["allowed"]:
            wrongful_allows += 1
        else:
            wrongful_denies += 1
        if not decision["allowed"]:
            print(f"deny {row['who']} {row['action']} {row['resource']}: {decision['reason']}")
    print(f"wrongful_allows {wrongful_allows}")
    print(f"wrongful_denies {wrongful_denies}")
    print(f"{matches} of {len(log)} match should_allow")
    return wrongful_allows, wrongful_denies, matches


wrongful_allows, wrongful_denies, matches = check_policy_on_log(ACTION_LOG)

deny ivy read_table salary_table: interns never see salary data
deny ivy send_email sales_table: no rule
deny ivy export_csv sales_table: no rule
deny ivy issue_refund refund_queue: no rule
deny ana read_table salary_table: no rule
deny ana export_csv sales_table: no rule
deny ana send_email customer_emails: only finance lead emails salary or customer data
deny ana read_table payroll_table: no rule
deny finn issue_refund refund_queue: no rule
deny finn delete_rows sales_table: no rule
deny adam send_email salary_table: only finance lead emails salary or customer data
deny adam send_email customer_emails: only finance lead emails salary or customer data
deny finn delete_rows salary_table: no rule
deny ivy read_table customer_emails: no rule
wrongful_allows 0
wrongful_denies 0
24 of 24 match should_allow


Every denial now names the rule that refused it, or says that no rule matched. The last three lines are the metric for the whole log.

## Step 7: Test the policy without calling anything

Each test checks one property the policy must keep, and none of them calls a model. The first two tests check the default and the rule order. Each test runs when its cell runs, and a failure stops the cell with the test name.

In [13]:
def test_default_is_deny():
    unknown = {"who": "ana", "action": "read_table", "resource": "payroll_table", "amount": None}
    assert decide_action(unknown, POLICY_RULES) == {"allowed": False, "reason": "no rule"}


def test_forbid_beats_permit():
    intern_salary = {"who": "ivy", "action": "read_table", "resource": "salary_table", "amount": None}
    assert find_matching_rules(intern_salary, POLICY_RULES)
    assert not decide_action(intern_salary, POLICY_RULES)["allowed"]


for test in [test_default_is_deny, test_forbid_beats_permit]:
    test()
    print(f"passed: {test.__name__}")

passed: test_default_is_deny
passed: test_forbid_beats_permit


The last three tests check the refund limit, the reasons and the whole log. They are kept in their own cell so each group prints its own results.

In [14]:
def test_refund_limit_edges():
    at_limit = {"who": "finn", "action": "issue_refund", "resource": "refund_queue", "amount": REFUND_LIMIT}
    over_limit = {"who": "finn", "action": "issue_refund", "resource": "refund_queue", "amount": REFUND_LIMIT + 1}
    assert decide_action(at_limit, POLICY_RULES)["allowed"]
    assert not decide_action(over_limit, POLICY_RULES)["allowed"]


def test_every_denial_has_a_reason():
    for row in ACTION_LOG:
        decision = decide_action(row, POLICY_RULES)
        if not decision["allowed"]:
            assert decision["reason"]


def test_policy_matches_the_whole_log():
    assert all(decide_action(row, POLICY_RULES)["allowed"] == row["should_allow"] for row in ACTION_LOG)


for test in [test_refund_limit_edges, test_every_denial_has_a_reason, test_policy_matches_the_whole_log]:
    test()
    print(f"passed: {test.__name__}")

passed: test_refund_limit_edges
passed: test_every_denial_has_a_reason
passed: test_policy_matches_the_whole_log


## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **Principal** | `USERS` | The person or program making a request |
| **Action** | `ACTIONS` | The operation the principal asks the agent to run |
| **Resource** | `RESOURCES` | The table, inbox or queue the action touches |
| **Permit and forbid** | `POLICY_RULES` and `FORBID_RULES` | A permit allows a match, and a forbid refuses it and wins |
| **Default deny** | `decide_action` | A request that matches no rule is refused |
| **Condition** | `within_amount_limit` | An extra test a rule must pass, such as an amount limit |
| **Decision reason** | the `reason` field from `decide_action` | The rule that decided the request, or "no rule" |